In [ ]:
from pathlib import Path
import json

import torch
from google.colab import drive

drive.mount("/content/drive")

drive_root = Path("/content/drive/MyDrive/FacialVisualProfile")
runs_root = drive_root / "runs"

shared_fields = [
    "seed",
    "epochs",
    "batch_size",
    "num_workers",
    "optimizer",
    "learning_rate",
    "weight_decay",
    "precision",
    "augmentation",
    "f1_threshold",
    "selection_metric",
    "epoch_seed_rule",
    "cudnn_deterministic",
    "cudnn_benchmark",
    "attribute_names",
]

configs = {}
for run_name in ("r18_seed0", "r50_seed0", "m3_seed0"):
    run_path = runs_root / run_name
    config = json.loads(
        (run_path / "config.json").read_text(encoding="utf-8")
    )
    best = json.loads(
        (run_path / "best.json").read_text(encoding="utf-8")
    )
    configs[run_name] = config

    print(f"\n=== {run_name} ===")
    print("Selected epoch:", best["epoch"])
    print("Selected checkpoint:", best["checkpoint"])
    print("Source commit:", config["git_commit"])
    print("Initial weights:", config["initial_weights"])

    checkpoint_path = run_path / best["checkpoint"]
    assert checkpoint_path.is_file()

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )
    assert checkpoint["epoch"] == best["epoch"]
    assert checkpoint["config"] == config

    state_keys = [
        key for key in ("model_state", "model_state_dict")
        if key in checkpoint
    ]
    assert len(state_keys) == 1
    state_key = state_keys[0]
    state = checkpoint[state_key]

    assert isinstance(state, dict)
    assert all(
        torch.isfinite(value).all().item()
        for value in state.values()
        if torch.is_tensor(value) and value.is_floating_point()
    )

    print("Weight field:", state_key)
    print("State tensor entries:", len(state))
    print("Selected checkpoint metadata and finite weights: PASS")

    # Inspect full history format before auditing selection rules.
    history_path = run_path / "history.json"
    if history_path.is_file():
        history = json.loads(history_path.read_text(encoding="utf-8"))
        assert isinstance(history, list) and len(history) == 10

        print("Full history: 10 epochs")
        print("First history record:")
        print(json.dumps(history[0], indent=2))
    else:
        print("Separate history.json: NOT FOUND")
        print("Checkpoint history length:", len(checkpoint.get("history", [])))

    del checkpoint, state

reference = configs["r18_seed0"]
print("\n=== Shared training protocol ===")
for field in shared_fields:
    assert field in reference
    for run_name, config in configs.items():
        assert field in config
        assert config[field] == reference[field], (
            f"Protocol mismatch: {run_name}, {field}"
        )
    if field == "attribute_names":
        print("Attribute order: identical, 24 attributes")
    else:
        print(f"{field}: {reference[field]}")

print("\nDay 13 protocol and checkpoint inspection: PASS")

In [ ]:
from pathlib import Path
import hashlib
import json
import math

drive_root = Path("/content/drive/MyDrive/FacialVisualProfile")
runs_root = drive_root / "runs"
report_dir = runs_root / "protocol_audit/day13"
report_dir.mkdir(parents=True, exist_ok=True)

shared_fields = [
    "seed", "epochs", "batch_size", "num_workers",
    "optimizer", "learning_rate", "weight_decay", "precision",
    "augmentation", "f1_threshold", "selection_metric",
    "epoch_seed_rule", "cudnn_deterministic", "cudnn_benchmark",
    "attribute_names",
]

run_configs = {}
audits = {}

for run_name in ("r18_seed0", "r50_seed0", "m3_seed0"):
    run_path = runs_root / run_name

    config = json.loads(
        (run_path / "config.json").read_text(encoding="utf-8")
    )
    best = json.loads(
        (run_path / "best.json").read_text(encoding="utf-8")
    )
    history = json.loads(
        (run_path / "history.json").read_text(encoding="utf-8")
    )
    run_configs[run_name] = config

    assert [row["epoch"] for row in history] == list(range(1, 11))

    if run_name == "m3_seed0":
        scores = [
            float(row["validation_mAP"]) for row in history
        ]
        saved_score = float(best["validation_mAP"])
    else:
        scores = [
            float(row["validation"]["map"]) for row in history
        ]
        saved_score = float(
            best["validation_map"]
            if run_name == "r18_seed0"
            else best["validation"]["map"]
        )

    assert all(math.isfinite(score) for score in scores)

    highest_score = max(scores)
    tied_epochs = [
        row["epoch"]
        for row, score in zip(history, scores)
        if score == highest_score
    ]
    selected_epoch = min(tied_epochs)

    assert best["epoch"] == selected_epoch
    assert best["checkpoint"] == f"epoch_{selected_epoch:02d}.pt"
    assert saved_score == highest_score

    # Check the fixed reporting threshold in every baseline epoch.
    if run_name != "m3_seed0":
        assert all(
            row["validation"]["threshold"] == config["f1_threshold"]
            for row in history
        )

    audits[run_name] = {
        "epochs_completed": len(history),
        "selected_epoch": selected_epoch,
        "selected_checkpoint": best["checkpoint"],
        "highest_validation_mAP": highest_score,
        "epochs_tied_for_highest_mAP": tied_epochs,
        "selection_matches_policy": True,
        "source_commit": config["git_commit"],
        "initial_weights": config["initial_weights"],
        "epoch_validation_mAP": [
            {"epoch": row["epoch"], "mAP": score}
            for row, score in zip(history, scores)
        ],
        "record_sha256": {
            filename: hashlib.sha256(
                (run_path / filename).read_bytes()
            ).hexdigest()
            for filename in ("config.json", "best.json", "history.json")
        },
    }

    print(
        f"{run_name}: completed=10, "
        f"selected=epoch_{selected_epoch:02d}, "
        f"mAP={highest_score:.9f}, "
        f"highest-score epochs={tied_epochs}; PASS"
    )

reference = run_configs["r18_seed0"]
for config in run_configs.values():
    for field in shared_fields:
        assert config[field] == reference[field]

report = {
    "shared_protocol": {
        field: reference[field] for field in shared_fields
    },
    "checkpoint_selection": {
        "metric": "validation_mAP",
        "direction": "maximize",
        "tie_rule": "earliest epoch among exactly equal maximum scores",
        "f1_used_for_selection": False,
    },
    "runs": audits,
    "scope": (
        "Audits saved configurations and all ten epoch records. "
        "Agreement with the tie policy does not demonstrate historical "
        "tie-handling execution when no tie occurred."
    ),
    "fresh_inference_check_in_this_notebook": "pending",
    "test_evaluated": False,
    "passed": True,
}

report_path = report_dir / "training_protocol_audit.json"
temporary = report_path.with_suffix(".json.tmp")
temporary.write_text(
    json.dumps(report, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)
temporary.replace(report_path)

print("Saved:", report_path)
print("Day 13 checkpoint-selection audit: PASS")

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
import io
import json
import sqlite3
import zipfile

import numpy as np
from PIL import Image
import torch

drive_root = Path("/content/drive/MyDrive/FacialVisualProfile")
runs_root = drive_root / "runs"
report_dir = runs_root / "protocol_audit/day13"

prediction_paths = {
    "r18_seed0": runs_root / "r18_seed0/day7/validation_predictions.npz",
    "r50_seed0": runs_root / "r50_seed0/day12/validation_predictions.npz",
    "m3_seed0": runs_root / "m3_seed0/validation_predictions.npz",
}
model_specs = {
    "r18_seed0": (
        "models/resnet18.py", "ResNet18Attributes", "model_state"
    ),
    "r50_seed0": (
        "models/resnet50.py", "ResNet50Attributes", "model_state"
    ),
    "m3_seed0": (
        "models/resnet18_geometry.py",
        "ResNet18GeometryAttributes",
        "model_state_dict",
    ),
}

def load_source_module(module_name, path):
    spec = importlib.util.spec_from_file_location(module_name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

# Load prediction records and verify alignment.
predictions = {}
for run_name, path in prediction_paths.items():
    with np.load(path, allow_pickle=False) as archive:
        predictions[run_name] = {
            key: archive[key].copy()
            for key in archive.files
        }

reference = predictions["m3_seed0"]
assert reference["geometry_valid"].dtype == np.bool_
assert reference["geometry_valid"].shape == (19_867,)
assert int((~reference["geometry_valid"]).sum()) == 32

for data in predictions.values():
    assert data["logits"].shape == (19_867, 24)
    for key in ("filenames", "targets", "attribute_names"):
        assert np.array_equal(data[key], reference[key])

indices = np.concatenate([
    np.flatnonzero(reference["geometry_valid"])[:3],
    np.flatnonzero(~reference["geometry_valid"])[:1],
])
assert len(indices) == 4
names = reference["filenames"][indices].tolist()
expected_valid = reference["geometry_valid"][indices]

# Read cached predicted coordinates for these four images.
cache_path = drive_root / "cache/mtcnn_clean.sqlite"
with sqlite3.connect(
    cache_path.resolve().as_uri() + "?mode=ro",
    uri=True,
) as connection:
    records = {}
    for name in names:
        row = connection.execute(
            "SELECT split, result FROM detections WHERE name = ?",
            (name,),
        ).fetchone()
        assert row is not None and row[0] == "1"
        records[name] = json.loads(row[1])

# Read original GT coordinates for baseline preprocessing.
gt_points = {}
with (
    drive_root / "data/list_landmarks_align_celeba.txt"
).open(encoding="utf-8-sig") as file:
    next(file)
    next(file)
    for line in file:
        name, *values = line.split()
        if name in names:
            gt_points[name] = torch.tensor(
                [float(value) for value in values],
                dtype=torch.float32,
            )
assert set(gt_points) == set(names)

# Read original image bytes and verify their hashes.
image_bytes = {}
with zipfile.ZipFile(
    drive_root / "data/img_align_celeba.zip"
) as archive:
    members = {}
    for member in archive.infolist():
        name = Path(member.filename).name
        if not member.is_dir() and name in names:
            assert name not in members
            members[name] = member

    assert set(members) == set(names)
    for name in names:
        content = archive.read(members[name])
        assert hashlib.sha256(content).hexdigest() == (
            records[name]["image_sha256"]
        )
        image_bytes[name] = content

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

atol = 1e-4
rtol = 1e-4
results = {}
reference_images = None

for run_name, (model_file, class_name, state_key) in model_specs.items():
    run_path = runs_root / run_name
    config = json.loads(
        (run_path / "config.json").read_text(encoding="utf-8")
    )
    best = json.loads(
        (run_path / "best.json").read_text(encoding="utf-8")
    )

    for filename, expected_hash in config["source_sha256"].items():
        source_path = run_path / "source" / filename
        assert hashlib.sha256(source_path.read_bytes()).hexdigest() == (
            expected_hash
        ), f"Source mismatch: {run_name}/{filename}"

    preprocessing = load_source_module(
        f"day13_{run_name}_preprocessing",
        run_path / "source/datasets/preprocessing.py",
    )
    model_module = load_source_module(
        f"day13_{run_name}_model",
        run_path / "source" / model_file,
    )

    images = []
    geometry = []
    for index, name in enumerate(names):
        with Image.open(io.BytesIO(image_bytes[name])) as image:
            assert list(image.size) == records[name]["image_size"]

            if run_name != "m3_seed0":
                image_tensor, _ = preprocessing.prepare(
                    image, gt_points[name]
                )
            elif expected_valid[index]:
                assert records[name]["status"] == "ok"
                raw_points = torch.tensor(
                    records[name]["landmarks"], dtype=torch.float32
                )
                assert raw_points.shape == (5, 2)
                assert torch.isfinite(raw_points).all()
                image_tensor, points = preprocessing.prepare(
                    image, raw_points
                )
                geometry.append(points)
            else:
                assert records[name]["status"] in (
                    "no_face", "invalid_output"
                )
                assert records[name]["landmarks"] is None
                image_tensor, _ = preprocessing.prepare(
                    image, torch.zeros(5, 2)
                )
                geometry.append(torch.zeros(10))

            images.append(image_tensor)

    images = torch.stack(images)
    if reference_images is None:
        reference_images = images.clone()
    else:
        assert torch.equal(images, reference_images)

    checkpoint_path = run_path / best["checkpoint"]
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )
    assert checkpoint["epoch"] == best["epoch"]
    assert checkpoint["config"] == config

    model_class = getattr(model_module, class_name)
    fresh_model = model_class(
        num_attributes=24, pretrained=False
    )
    fresh_model.load_state_dict(checkpoint[state_key], strict=True)
    fresh_model.to(device).eval()
    del checkpoint

    with torch.inference_mode():
        if run_name == "m3_seed0":
            logits = fresh_model(
                images.to(device),
                torch.stack(geometry).to(device),
                torch.tensor(
                    expected_valid, dtype=torch.bool, device=device
                ),
            )
        else:
            logits = fresh_model(images.to(device))

    assert logits.shape == (4, 24)
    assert torch.isfinite(logits).all()

    actual = logits.cpu().numpy()
    expected = predictions[run_name]["logits"][indices]
    difference = float(np.max(np.abs(actual - expected)))
    passed = bool(np.allclose(actual, expected, atol=atol, rtol=rtol))

    results[run_name] = {
        "checkpoint": best["checkpoint"],
        "source_commit": config["git_commit"],
        "saved_source_hashes_verified": True,
        "strict_state_loading": True,
        "logits_shape": list(actual.shape),
        "maximum_absolute_logit_difference": difference,
        "matches_saved_predictions": passed,
    }
    print(
        f"{run_name}: logits={actual.shape}, "
        f"max difference={difference:.8f}; "
        f"{'PASS' if passed else 'FAIL'}"
    )

    del fresh_model, logits
    if device.type == "cuda":
        torch.cuda.empty_cache()

passed = all(
    result["matches_saved_predictions"]
    for result in results.values()
)
report = {
    "sample_filenames": names,
    "geometry_valid": expected_valid.tolist(),
    "image_hashes_verified": True,
    "image_tensors_equal_across_models": True,
    "device": str(device),
    "torch": str(torch.__version__),
    "absolute_tolerance": atol,
    "relative_tolerance": rtol,
    "models": results,
    "scope": (
        "Four-sample fresh-model reload check against saved validation "
        "logits, including one failed detection. Not a new full-validation "
        "evaluation or a training run."
    ),
    "test_evaluated": False,
    "passed": passed,
}

report_dir.mkdir(parents=True, exist_ok=True)
report_path = report_dir / "three_model_reload_report.json"
temporary = report_path.with_suffix(".json.tmp")
temporary.write_text(
    json.dumps(report, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)
temporary.replace(report_path)

print("Samples:", names)
print("Geometry valid:", expected_valid.tolist())
print("Saved:", report_path)

assert passed, "Reload outputs differ; inspect the report before proceeding."

audit_path = report_dir / "training_protocol_audit.json"
audit = json.loads(audit_path.read_text(encoding="utf-8"))
audit["fresh_inference_check_in_this_notebook"] = "passed"
audit["fresh_inference_report"] = report_path.name
temporary_audit = audit_path.with_suffix(".json.tmp")
temporary_audit.write_text(
    json.dumps(audit, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)
temporary_audit.replace(audit_path)

print("Day 13 three-model reload check: PASS")

In [ ]:
from pathlib import Path
import json
import zipfile

from google.colab import files

report_dir = Path(
    "/content/drive/MyDrive/FacialVisualProfile"
    "/runs/protocol_audit/day13"
)
filenames = [
    "training_protocol_audit.json",
    "three_model_reload_report.json",
]

for name in filenames:
    report = json.loads(
        (report_dir / name).read_text(encoding="utf-8")
    )
    assert report["passed"] is True

audit = json.loads(
    (report_dir / filenames[0]).read_text(encoding="utf-8")
)
assert audit["fresh_inference_check_in_this_notebook"] == "passed"

archive_path = Path("/content/day13_results.zip")
with zipfile.ZipFile(
    archive_path, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for name in filenames:
        archive.write(report_dir / name, arcname=name)

files.download(str(archive_path))